In [1]:
!pip install pulp

#ライブラリのインポート

import pulp 
import time
import pandas as pd
import numpy as np

     |████████████████████████████████| 40.6MB 77kB/s 


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#データの読み込み

import glob
import os
import pickle

INPUT_DIR = "/content/drive/My Drive/Google Colab/takano_Lab2/scenarios/data_7"
INPUT_CONST_DIR = "/content/drive/My Drive/Google Colab/takano_Lab2/scenarios/Constant"

#読みこみ
def pickle_load(path):
    with open(path, mode='rb') as f:
        data = pickle.load(f)
        return data

#書き出し
def pickle_dump(obj, path):
    with open(path, mode='wb') as f:
        pickle.dump(obj,f)

def import_data_dfm(y,m):
  csv_files_dir_start = sorted(glob.glob(os.path.join(INPUT_DIR, 'start/{0}/start_trip_{0}0{1}*'.format(y,m))))
  marge_csv_start = [pd.read_csv(f, encoding='cp932') for f in csv_files_dir_start]
  df_master_start = pd.concat(marge_csv_start, ignore_index=True)

  csv_files_dir_end = sorted(glob.glob(os.path.join(INPUT_DIR, 'end/{0}/end_trip_{0}0{1}*'.format(y,m))))
  marge_csv_end = [pd.read_csv(f, encoding='cp932') for f in csv_files_dir_end]
  df_master_end = pd.concat(marge_csv_end, ignore_index=True)
  sce_num = len(csv_files_dir_start)

  return df_master_start,df_master_end, sce_num

df_master_start,df_master_end, sce_num = import_data_dfm(2015,4)

station_C = pd.read_csv((os.path.join(INPUT_CONST_DIR, 'station_C.csv')))
K = pd.read_csv((os.path.join(INPUT_CONST_DIR, 'trip_duration.csv')))

In [ ]:
#各種値の指定
station_num = 7
time_num = 288
scenario_num = sce_num
time_span = 1
time_start = 8
time_end = 20

#集合の生成
#ここでは8:00~20:00まで1時間ごとに計13回のリバランスを考える
D = list(range(1,station_num+1))
T = list(range(1,time_num+1))
T_ = list(range(1,time_num+2))
S = list(range(1,scenario_num+1))
T2hour = list((time*12*time_span)+1 for time in range(int(time_start/time_span),int(time_end/time_span)+1))

STJ = [(s,t,j) for j in D for t in T_ for s in S]
STIJ = [(s,t,i,j) for j in D for i in D for t in T for s in S]
TIJ = [(t,i,j) for j in D for i in D for t in T]
SJ = [(s,j) for j in D for s in S]
IJ = [(i,j) for j in D for i in D]
ST2hourIJ = [(s,t,i,j) for j in D for i in D for t in T2hour for s in S]
T2hourIJ = [(t,i,j) for j in D for i in D for t in T2hour]
ST2hourJ = [(s,t,j) for j in D for t in T2hour for s in S]

print(D)
print(T)
print(S)

s_trip = {}
e_trip = {}

for stij in STIJ:
    s_trip[stij] = 0 
    e_trip[stij] = 0

for s in S:
  n = 288*(s-1)
  for t in T:
    for i in D:
      for j in D:
        s_trip[s,t,i,j] = int(df_master_start.iloc[[n+t-1]]['{0}_{1}'.format(i-1,j-1)])         
        e_trip[s,t,i,j] = int(df_master_end.iloc[[n+t-1]]['{0}_{1}'.format(i-1,j-1)])  

[1, 2, 3, 4, 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 

In [ ]:
def culc_average_s(s_trip):
  s_trip_ave_s = {}
  for tij in TIJ:
    s_trip_ave_s[tij] = 0
  for t in T:
    for i in D:
      for j in D:
        sum_trips = 0
        for s in S:
          sum_trips += int(s_trip[s,t,i,j])
        s_trip_ave_s[t,i,j] = sum_trips/scenario_num

  return s_trip_ave_s

def culc_average_st(s_trip):
  s_trip_ave_st = {}
  for ij in IJ:
    s_trip_ave_st[ij] = 0
  for i in D:
    for j in D:
      sum_trips = 0
      for s in S:
        for t in T:
          sum_trips += int(s_trip[s,t,i,j])
      s_trip_ave_st[i,j] = sum_trips/scenario_num/time_num

  return s_trip_ave_st

def culc_average_s_2hour(s_trip):
  s_trip_ave_s_2hour = {}
  for tij in T2hourIJ:
    s_trip_ave_s_2hour[tij] = 0
  for t in T2hour:
    for i in D:
      for j in D:
        sum_trips = 0
        for s in S:
          for n in range(t-12,t):
            sum_trips += int(s_trip[s,n,i,j])
        s_trip_ave_s_2hour[t,i,j] = sum_trips/scenario_num

  return s_trip_ave_s_2hour

def culc_trip_dif(s_trip, e_trip):
  trip_dif = {}
  trip_d = {}
  for stj in ST2hourJ:
    trip_dif[stj] = 0
    trip_d[stj] = 0
  for s in S:
    for t in T2hour:
      for j in D:
        trip_dpt = 0
        trip_arr = 0
        for n in range(t-12,t):
          for l in D:
            trip_dpt += int(s_trip[s,n,j,l])
            trip_arr += int(e_trip[s,n,l,j])
        trip_dif[s,t,j] = (trip_arr-trip_dpt)
        trip_d[s,t,j] = trip_dpt

  return trip_dif, trip_d

s_trip_ave_s = culc_average_s(s_trip)
s_trip_ave_st = culc_average_st(s_trip)
s_trip_ave_s_2hour = culc_average_s_2hour(s_trip)
trip_dif,trip_d = culc_trip_dif(s_trip,e_trip)

In [ ]:
#クラスタごとの上限値を作成
C = station_C.groupby('cluster_id')['dock_count'].agg(['sum'])
C2M = {}
i = 1
for row in C.itertuples():
    C2M[i] = row.sum
    i += 1
C2M

#駅間のトリップ時間を辞書に格納
K2D = {}

for i in D:
  for j in D:
    if i == j:
      K2D[i,j] = 0
    else:
      K2D[i,j] = K.iat[i-1,j-1]

In [ ]:
def culc_trip_dif(s_trip, e_trip):
  trip_dif = {}
  trip_d = {}
  for stj in ST2hourJ:
    trip_dif[stj] = 0
    trip_d[stj] = 0
  for s in S:
    for t in T2hour:
      for j in D:
        trip_dpt = 0
        trip_arr = 0
        for n in range(t-12,t):
          for l in D:
            trip_dpt += int(s_trip[s,n,j,l])
            trip_arr += int(e_trip[s,n,l,j])
        trip_dif[s,t,j] = (trip_arr-trip_dpt)
        trip_d[s,t,j] = trip_dpt

  return trip_dif, trip_d

trip_dif,trip_d = culc_trip_dif(s_trip,e_trip)

In [ ]:
#リバランス部分の作成

class MobilitySolver:
      def __init__(self):
        pass
    
      def make_rebalence(self,name,weight):
          self.r = pickle_load('/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/r_{0}{1}_201503.pickle'.format(name,weight))
          self.w = pickle_load('/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/w_{0}{1}_201503.pickle'.format(name,weight))
          self.z_i = pickle_load('/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201503_train/z_{0}{1}_201503.pickle'.format(name,weight))

          self.r_s = {}
          for stij in ST2hourIJ:
            self.r_s[stij] = 0

          #2way
          if name == '2dir':
            for s in S:
              for t in T2hour:
                  for i in D:
                      for j in D:
                        self.trip_2way = 0
                        for n in range(t-12,t):
                          self.trip_2way += (s_trip[s,n,i,j]-s_trip[s,n,j,i])
                        if self.w[t,i,j].value() == None:
                          self.r_s[s,t,i,j] = self.r[t,i,j].value()
                        else:
                          self.r_s[s,t,i,j] = self.r[t,i,j].value() - self.w[t,i,j].value()*self.trip_2way
          
          elif name == 'streb':
            for s in S:
              for t in T2hour:
                  for i in D:
                      for j in D:
                        self.r_s[s,t,i,j] = self.r[t,i,j].value()

          elif name == 'noreb':
            for s in S:
              for t in T2hour:
                  for i in D:
                      for j in D:
                        self.r_s[s,t,i,j] = 0
                        self.r[t,i,j] = 0

          elif name == 'dpt_arr':
            for s in S:
              for t in T2hour:
                for i in D:
                  for j in D:
                    if self.w[t,i,j].value() == None:
                      self.r_s[s,t,i,j] = self.r[t,i,j].value()
                    else:
                      self.r_s[s,t,i,j] = self.r[t,i,j].value() + self.w[t,i,j].value()*(trip_dif[s,t,i]-trip_dif[s,t,j])

      def solve(self,weight_para):

          def func_r_s_2hour(s,t,i,j):
            if t-K.iat[i-1,j-1] in T2hour:
              t_past = t-K.iat[i-1,j-1]
              return self.r_s[s,t_past,i,j]
            else:
              return 0

          #動的リバランスを考慮
          alpha = 1
          beta = 1 #正則化項
          gamma = weight_para
          delta = 1

          # 時間計測開始
          time_start = time.perf_counter()

          problem = pulp.LpProblem('BikeShare',pulp.LpMinimize)

          #決定変数を定義する

          z = pulp.LpVariable.dicts('z', STJ,lowBound=0, cat='Continuous')
          a = pulp.LpVariable.dicts('a', STJ,lowBound=0, cat='Continuous')
          x = pulp.LpVariable.dicts('x', SJ,lowBound=0, cat='Continuous')
          y = pulp.LpVariable.dicts('y', SJ,lowBound=0, cat='Continuous')

          #目的関数を宣言する
          problem += alpha*pulp.lpSum(K2D[i,j]*self.r_s[s,t,i,j] for j in D for i in D for t in T2hour for s in S)/scenario_num + beta*pulp.lpSum(a[s,t,j] for j in D for t in T_ for s in S)/scenario_num + gamma*(pulp.lpSum(self.w[t,i,j] for j in D for i in D for t in T2hour)) + delta*pulp.lpSum(x[s,j]+y[s,j] for j in D for s in S)/scenario_num


          for s in S:

            #絶対値の処理
            for j in D:
              problem += z[s,289,j]- z_i[s,1,j] == x[s,j]-y[s,j]

            #problem += sum(z_i[j] for j in D) == 329

            for j in D:
              problem += z[s,1,j] == self.z_i[s,1,j]

            # 制約条件① 駅の車両台数の推移
            for t in T:
              for j in D:
                if t in T2hour:
                  problem += z[s,t+1,j] == z[s,t,j] - sum(s_trip[s,t,j,i] for i in D) - sum(self.r_s[s,t,j,i] for i in D) + sum(e_trip[s,t,i,j] for i in D)
                else:
                  problem += z[s,t+1,j] == z[s,t,j] - sum(s_trip[s,t,j,i] for i in D) + sum(e_trip[s,t,i,j] for i in D) + sum(func_r_s_2hour(s,t,i,j) for i in D)
              
            # 制約条件②
            for t in T2hour:
                for i in D:
                    problem += self.r_s[s,t,i,i] == 0
                    problem += self.w[t,i,i] == 0
                    problem += self.r[t,i,i] == 0

            # 制約条件② 
            for t in T:
                for j in D:
                  if t in T2hour:
                    problem += z[s,t,j] >= (sum(s_trip[s,t,j,i] for i in D) + sum(self.r_s[s,t,j,i] for i in D))
                  else:
                    problem += z[s,t,j] >= sum(s_trip[s,t,j,i] for i in D)

            # 制約条件③ 初期値からの乖離
            for t in T_:
                for j in D:
                    problem += z[s,t,j] <= C2M[j] + a[s,t,j]
                    problem += z[s,t,j] >= 0 - a[s,t,j]

          status = problem.solve()
          # 時間計測終了
          time_stop = time.perf_counter()      


          self.Z = [[[0 for j in range(station_num)]for t in range(time_num+1)] for s in range(scenario_num)]
          A = [[[0 for j in range(station_num)]for t in range(time_num+1)] for s in range(scenario_num)]
          XY = [[0 for j in range(station_num)]for s in range(scenario_num)]

          sum_Rs = 0
          sum_Rs_K = 0
          for s in S:
              for i in D:
                  for j in D:
                      for t in T2hour:
                          sum_Rs += self.r_s[s,t,i,j]
                          sum_Rs_K += self.r_s[s,t,i,j]*K2D[i,j]

          for s in range(scenario_num):
              for j in range(station_num):
                  for t in range(time_num+1):
                      self.Z[s][t][j] = z[s+1,t+1,j+1].value()

          sum_A = 0
          for s in range(scenario_num):
              for j in range(station_num):
                  for t in range(time_num+1):
                      A[s][t][j] = a[s+1,t+1,j+1].value()
                      sum_A += a[s+1,t+1,j+1].value()

          sum_W = 0
          for i in D:
            for j in D:
              for t in T2hour:
                if w[t,i,j].value() == None:
                  pass
                else:
                  sum_W += w[t,i,j].value() 

          sum_R = 0
          sum_R_K = 0
          for i in D:
            for j in D:
              for t in T2hour:
                if self.r[t,i,j] != 0:
                  sum_R += self.r[t,i,j].value()
                  sum_R_K += self.r[t,i,j].value()*K2D[i,j]

          sum_XY = 0
          for s in range(scenario_num):
              for j in range(station_num):
                    XY[s][j] = x[s+1,j+1].value()+y[s+1,j+1].value()
                    sum_XY += x[s+1,j+1].value()+y[s+1,j+1].value()  


          self.obj = pulp.value(problem.objective)
          self.R_s_sce = sum_Rs/scenario_num
          self.R = sum_R
          self.R_s_sce_K = sum_Rs_K/scenario_num
          self.R_K = sum_R_K   
          self.A_sce = sum_A/scenario_num
          self.XY_sce = sum_XY/scenario_num
          self.W = sum_W
          self.time = time_stop-time_start
          self.status = pulp.LpStatus[status]

          return self.r, self.w, z, self.z_i, a, self.r_s, x, y 

In [ ]:
NAME_WEIGHT_LIST = [['noreb','',1],['streb','',1],
                    ['2dir','W0',0],['2dir','W0.5',0.5],['2dir','W1',1],['2dir','W1.5',1.5],['2dir','W2',2],['2dir','W2.5',2.5],['2dir','W3',3],['2dir','W3.5',3.5],['2dir','W4',4],['2dir','W4.5',4.5],['2dir','W5',5],
                    ['dpt_arr','W0',0],['dpt_arr','W0.5',0.5],['dpt_arr','W1',1],['dpt_arr','W1.5',1.5],['dpt_arr','W2',2],['dpt_arr','W2.5',2.5],['dpt_arr','W3',3],['dpt_arr','W3.5',3.5],['dpt_arr','W4',4],['dpt_arr','W4.5',4.5],['dpt_arr','W5',5]]

Rows = []
for name, weight, weight_para in NAME_WEIGHT_LIST:
  print('===', name, weight, weight_para, '===')

  ms = MobilitySolver()
  ms.make_rebalence(name,weight)
  r, w, z, z_i, a, r_s, x, y =  ms.solve(weight_para)

  row = (name, weight_para, ms.R_s_sce, ms.R_s_sce_K, ms.R, ms.R_K, ms.A_sce, ms.XY_sce, ms.W, ms.R_s_sce_K+ms.A_sce+ms.XY_sce, ms.obj, ms.time, ms.status)
  Rows.append(row)

  pickle_dump(r, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/r_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(w, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/w_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(z, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/z_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(z_i, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/z_i_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(a, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/a_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(r_s, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/r_s_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(x, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/x_{0}{1}_201504.pickle'.format(name,weight))
  pickle_dump(y, '/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/y_{0}{1}_201504.pickle'.format(name,weight))

dfm = pd.DataFrame(Rows, columns = ['name','gamma','R_s/sce','R_s*K/sce','R','R*K','A/sce','|Z[289]-Z[1]|/sce','W','R_s*K/sce+A/sce+|Z[289]-Z[1]|/sce','objective','time','status'])

=== noreb  1 ===
=== streb  1 ===
=== 2dir W0 0 ===
=== 2dir W0.5 0.5 ===
=== 2dir W1 1 ===
=== 2dir W3 3 ===
=== ave W0 0 ===
=== ave W0.5 0.5 ===
=== ave W1 1 ===
=== dpt_arr W0 0 ===
=== dpt_arr W0.5 0.5 ===
=== dpt_arr W1 1 ===
=== dpt_arr W3 3 ===
=== dpt W0 0 ===
=== dpt W0.5 0.5 ===
=== dpt W1 1 ===
=== dpt W3 3 ===


In [ ]:
dfm.to_csv('/content/drive/My Drive/Google Colab/takano_Lab2/output_thesis/201504_test/dfm_201504.csv',index=None)
dfm

,name,gamma,R_s/sce,R,A/sce,|Z[288]-Z[1]|/sce,W,R_s/sce+A/sce+|Z[288]-Z[1]|/sce,objective,time,status
0,noreb,1.0,0.000000,0.000000,4006.300000,88.566667,0.000000,4094.866667,4094.866667,27.504762,Optimal
1,streb,1.0,69.000000,69.000000,961.000000,92.433333,0.000000,1122.433333,1161.433333,27.125134,Optimal
2,2dir,0.0,64.851680,39.906417,2262.222706,68.681496,16.780318,2395.755881,2455.816040,27.356072,Optimal
3,2dir,0.5,63.192074,37.799131,2329.093198,68.815604,0.000000,2461.100876,2521.954608,27.229894,Optimal
4,2dir,1.0,62.250833,36.745972,2310.783517,68.607552,0.000000,2441.641902,2502.765433,27.242535,Optimal
5,2dir,3.0,61.708189,37.374090,2172.820243,72.503931,0.000000,2307.032362,2365.859936,27.639296,Optimal
6,ave,0.0,121.452859,121.452859,2944.346002,103.950489,26.988533,3169.749350,3258.792567,27.016345,Optimal
7,ave,0.5,116.903999,116.903999,2801.241611,104.597809,0.000000,3022.743418,3113.704271,27.095237,Optimal
8,ave,1.0,111.710101,111.710101,2773.946187,104.129479,0.000000,2989.785767,3083.067066,27.388086,Optimal
9,dpt_arr,0.0,67.692069,43.303630,551.243749,71.293587,5.269568,690.229405,731.310966,27.356702,Optimal
